# DSS_Variables_View — long-format projection over cosmos-graph Variables

Produces `consumer-bases/interim/DSS_Variables_View.xlsx` with two sheets:

| Sheet | Grain | Source |
|---|---|---|
| `ReadMe` | — | provenance and column dictionary |
| `Variables` | one row per Variables row | `cosmos-graph` Variables joined to DSS (for `domain`), SDTM_Domain_Metadata (for `observation_class`), and AssignedTerms (for pinned-value preferred term) |

## Why

Long-format complement to `DSS_View.xlsx Measurement_Specs`. The wide pivot is the canonical lookup for pinned values at DSS grain; consumers needing per-variable detail (value_list breadth, bare codelist bindings, role, mandatory flags, origin) join here on `ds_id`.

Closes the recurring "drop into the Variables sheet for value_list breadth" friction documented from the X-ray case study onwards.

## Inputs

| File | Track | Sheets used |
|---|---|---|
| `cosmos-graph/interim/COSMoS_Graph.xlsx` | cosmos-graph | DSS, Variables |
| `cosmos-graph/interim/COSMoS_Graph_CT.xlsx` | cosmos-graph | AssignedTerms |
| `sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx` | sdtm-domain-reference | Domains |

## Output

`consumer-bases/interim/DSS_Variables_View.xlsx`

## Join model

- **Primary key.** `(ds_id, variable_name)` — every pair is unique in the source.
- **Foreign key to DSS_View.** `ds_id` — one DSS_View.Measurement_Specs row to many DSS_Variables_View.Variables rows.

## 1. Setup

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

In [ ]:
BASE_DIR = Path.cwd().parent          # consumer-bases/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/

GRAPH_FILE = REPO_ROOT / 'cosmos-graph' / 'interim' / 'COSMoS_Graph.xlsx'
GRAPH_CT_FILE = REPO_ROOT / 'cosmos-graph' / 'interim' / 'COSMoS_Graph_CT.xlsx'
DOMAIN_META_FILE = REPO_ROOT / 'sdtm-domain-reference' / 'machine_actionable' / 'SDTM_Domain_Metadata.xlsx'

INTERIM_DIR = BASE_DIR / 'interim'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = INTERIM_DIR / 'DSS_Variables_View.xlsx'

for f, label in [
    (GRAPH_FILE, 'Graph'),
    (GRAPH_CT_FILE, 'Graph CT'),
    (DOMAIN_META_FILE, 'Domain Meta'),
]:
    if f.exists():
        print(f'  {label}: {f.relative_to(REPO_ROOT)}')
    else:
        raise FileNotFoundError(f'{label} file not found: {f}')

print(f'  Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')

## 2. Load inputs

In [ ]:
vars_g = pd.read_excel(GRAPH_FILE, sheet_name='Variables', dtype=str).fillna('')
dss_g = pd.read_excel(GRAPH_FILE, sheet_name='DSS', dtype=str).fillna('')
assigned_terms = pd.read_excel(GRAPH_CT_FILE, sheet_name='AssignedTerms', dtype=str).fillna('')
domain_meta = pd.read_excel(DOMAIN_META_FILE, sheet_name='Domains', dtype=str).fillna('')

print(f'Variables:      {len(vars_g):>6,} rows')
print(f'DSS:            {len(dss_g):>6,} rows')
print(f'AssignedTerms:  {len(assigned_terms):>6,} rows  (CT-resolved pinned-value enrichment)')
print(f'Domains:        {len(domain_meta):>6,} rows  (reference)')

## 3. Build Variables sheet

Project the source Variables row to the consumer-relevant columns, join three reference sources, and verify the join cardinalities are 1:1.

In [ ]:
# Project Variables to consumer-relevant columns
PROJECT_COLS = [
    'ds_id', 'bc_id', 'variable_name',
    'role', 'data_type', 'mandatory_variable', 'mandatory_value',
    'origin_type', 'origin_source',
    'assigned_term_value', 'assigned_term_concept_id',
    'value_list',
    'codelist_concept_id', 'codelist_submission_value',
    'subset_codelist', 'vlm_target',
]

missing = [c for c in PROJECT_COLS if c not in vars_g.columns]
if missing:
    raise RuntimeError(f'Variables sheet missing columns: {missing}')

view = vars_g[PROJECT_COLS].copy()
print(f'Variables (projected): {len(view):,} rows x {len(view.columns)} cols')

In [ ]:
# Join AssignedTerms for the NCIt preferred term of any pinned value
at_lookup = assigned_terms[['assigned_term_concept_id', 'term_nci_preferred_term']].rename(
    columns={'term_nci_preferred_term': 'assigned_term_preferred_term'}
)
before = len(view)
view = view.merge(at_lookup, on='assigned_term_concept_id', how='left').fillna('')
if len(view) != before:
    raise RuntimeError(f'AssignedTerms join changed row count: {before} -> {len(view)}')

# Sanity: rows with assigned_term_concept_id set but preferred_term unresolved
pinned = view[view['assigned_term_concept_id'] != '']
unresolved = (pinned['assigned_term_preferred_term'] == '').sum()
print(f'Pinned rows: {len(pinned):,}  (unresolved against AssignedTerms: {unresolved})')

In [ ]:
# Join DSS for domain
ds_dom = dss_g[['ds_id', 'domain']].drop_duplicates()
before = len(view)
view = view.merge(ds_dom, on='ds_id', how='left').fillna('')
if len(view) != before:
    raise RuntimeError(f'DSS join changed row count: {before} -> {len(view)}')
missing_dom = (view['domain'] == '').sum()
if missing_dom:
    raise RuntimeError(f'{missing_dom} Variables rows have no DSS.domain')
print(f'Domains joined cleanly. Distinct domains: {view["domain"].nunique()}')

In [ ]:
# Join SDTM_Domain_Metadata for observation_class
dom_lookup = domain_meta[['Domain', 'Observation_Class']].rename(columns={
    'Domain': 'domain', 'Observation_Class': 'observation_class',
})
before = len(view)
view = view.merge(dom_lookup, on='domain', how='left').fillna('')
if len(view) != before:
    raise RuntimeError(f'Domain metadata join changed row count: {before} -> {len(view)}')
missing_oc = (view['observation_class'] == '').sum()
if missing_oc:
    raise RuntimeError(f'{missing_oc} Variables rows have no observation_class')
print(f'observation_class joined cleanly. Distribution:')
print(view['observation_class'].value_counts())

In [ ]:
# Final column order
FINAL_COLS = [
    'ds_id', 'variable_name', 'bc_id',
    'domain', 'observation_class',
    'role', 'data_type', 'mandatory_variable', 'mandatory_value',
    'origin_type', 'origin_source',
    'assigned_term_value', 'assigned_term_concept_id', 'assigned_term_preferred_term',
    'value_list',
    'codelist_concept_id', 'codelist_submission_value', 'subset_codelist',
    'vlm_target',
]

missing = [c for c in FINAL_COLS if c not in view.columns]
extra = [c for c in view.columns if c not in FINAL_COLS]
if missing or extra:
    raise RuntimeError(f'Column mismatch. Missing: {missing}. Extra: {extra}')

variables_out = view[FINAL_COLS].sort_values(['ds_id', 'variable_name']).reset_index(drop=True)
print(f'Variables sheet: {len(variables_out):,} rows x {len(variables_out.columns)} cols')
variables_out.head()

## 4. Write workbook

Two sheets: `ReadMe`, `Variables`. Color convention follows the repo standard:

- **Grey** headers — keys and foreign keys (ds_id, variable_name, bc_id, codelist_concept_id).
- **Yellow** headers — COSMoS-side metadata and graph-CT enrichment (AssignedTerms preferred term).
- **Green** headers — reference-side enrichment (observation_class from SDTM_Domain_Metadata).

In [ ]:
# Styles
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
WRAP = Alignment(wrap_text=True, vertical='top')

GREEN_HEADER = PatternFill('solid', fgColor='548235')   # TESTCD / SDTM CT side
YELLOW_HEADER = PatternFill('solid', fgColor='FFD700')  # COSMoS side
GREY_HEADER = PatternFill('solid', fgColor='808080')    # keys, aggregation


def write_sheet(ws, df, header_fills, col_widths):
    cols = list(df.columns)
    for ci, name in enumerate(cols, 1):
        cell = ws.cell(row=1, column=ci, value=name)
        cell.font = HEADER_FONT
        cell.fill = header_fills.get(name, GREY_HEADER)
        cell.alignment = WRAP
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, name in enumerate(cols, 1):
            val = row[name]
            cell = ws.cell(row=ri, column=ci, value=val if val != '' else None)
            cell.font = DATA_FONT
            cell.alignment = WRAP
    for ci, name in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(ci)].width = col_widths.get(name, 18)
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = f'A1:{get_column_letter(len(cols))}1'


print('Writer ready.')

In [ ]:
wb = Workbook()

# ── ReadMe sheet ──
ws_rm = wb.active
ws_rm.title = 'ReadMe'

readme_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', size=12, bold=True)
section_font = Font(name='Arial', size=10, bold=True)

readme_lines = [
    ('DSS_Variables_View — long-format projection over Variables', title_font),
    ('', None),
    ('PROVENANCE', section_font),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', readme_font),
    (f'Notebook: consumer-bases/notebooks/20_dss_variables_view.ipynb', readme_font),
    (f'Inputs:', readme_font),
    (f'  cosmos-graph/interim/COSMoS_Graph.xlsx (DSS, Variables)', readme_font),
    (f'  cosmos-graph/interim/COSMoS_Graph_CT.xlsx (AssignedTerms)', readme_font),
    (f'  sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx (Domains)', readme_font),
    ('', None),
    ('SCOPE', section_font),
    ('Long-format complement to DSS_View.xlsx Measurement_Specs.', readme_font),
    ('Graph-wide. Every Variables row across all 32 domains, not only Findings.', readme_font),
    ('Observation-class scoping is applied by each consumer track.', readme_font),
    ('', None),
    ('SCOPE DISCIPLINE', section_font),
    ('Joined, denormalised projection of cosmos-graph plus repo reference', readme_font),
    ('metadata. No interpretive overlays — sub-typing, behavioural classification,', readme_font),
    ('and narrative framing belong to the consumer track.', readme_font),
    ('', None),
    ('JOIN MODEL', section_font),
    ('Primary key:        (ds_id, variable_name)', readme_font),
    ('Foreign key to', readme_font),
    ('  DSS_View:         ds_id  (one Measurement_Specs row to many here)', readme_font),
    ('  Codelists:        codelist_concept_id', readme_font),
    ('  Codelist_Coverage:codelist_concept_id', readme_font),
    ('', None),
    ('SHEET — Variables', section_font),
    ('  ds_id, variable_name        — primary key', readme_font),
    ('  bc_id                       — parent Biomedical Concept', readme_font),
    ('  domain                      — SDTM domain code', readme_font),
    ('  observation_class           — Findings / Events / Interventions / etc.', readme_font),
    ('  role                        — Qualifier / Timing / Topic / Identifier', readme_font),
    ('  data_type                   — text / float / integer / datetime / ...', readme_font),
    ('  mandatory_variable          — Y / N', readme_font),
    ('  mandatory_value             — Y / N', readme_font),
    ('  origin_type                 — Collected / Assigned / Derived / blank', readme_font),
    ('  origin_source               — Investigator / Sponsor / Vendor / blank', readme_font),
    ('  assigned_term_value         — pinned submission value (if any)', readme_font),
    ('  assigned_term_concept_id    — NCIt code of pinned value (if any)', readme_font),
    ('  assigned_term_preferred_term— NCIt preferred term, joined from AssignedTerms', readme_font),
    ('  value_list                  — verbatim value_list string (if any)', readme_font),
    ('  codelist_concept_id         — bound codelist NCIt code (if any)', readme_font),
    ('  codelist_submission_value   — bound codelist submission value (if any)', readme_font),
    ('  subset_codelist             — CDISC subset reference (e.g. NY_NY)', readme_font),
    ('  vlm_target                  — Y if this row is a value-list-metadata target', readme_font),
    ('', None),
    ('BINDING MODES', section_font),
    ('A Variables row that carries a codelist is in one of three modes,', readme_font),
    ('mutually exclusive at row level:', readme_font),
    ('  pinned       — assigned_term_value non-empty', readme_font),
    ('  value_list   — value_list non-empty (and not pinned)', readme_font),
    ('  bare         — codelist bound, neither pinned nor value_list', readme_font),
    ('Bare bindings drop out of DSS_View.Measurement_Specs (which keys on', readme_font),
    ('slot pins). They are preserved here.', readme_font),
    ('', None),
    ('STATUS', section_font),
    ('Long-format projection. Joinable to DSS_View on ds_id (one-to-many).', readme_font),
    ('Sources: COSMoS public exports + NCI EVS CT package 2026-03-27.', readme_font),
]

for ri, (text, font) in enumerate(readme_lines, 1):
    cell = ws_rm.cell(row=ri, column=1, value=text if text else None)
    if font:
        cell.font = font

ws_rm.column_dimensions['A'].width = 100
print(f'ReadMe: {len(readme_lines)} lines')

In [ ]:
# ── Variables sheet ──
ws_v = wb.create_sheet('Variables')

V_FILLS = {
    # Keys / foreign keys — grey
    'ds_id':                       GREY_HEADER,
    'variable_name':               GREY_HEADER,
    'bc_id':                       GREY_HEADER,
    'codelist_concept_id':         GREY_HEADER,
    # COSMoS source — yellow
    'domain':                      YELLOW_HEADER,
    'role':                        YELLOW_HEADER,
    'data_type':                   YELLOW_HEADER,
    'mandatory_variable':          YELLOW_HEADER,
    'mandatory_value':             YELLOW_HEADER,
    'origin_type':                 YELLOW_HEADER,
    'origin_source':               YELLOW_HEADER,
    'assigned_term_value':         YELLOW_HEADER,
    'assigned_term_concept_id':    YELLOW_HEADER,
    'assigned_term_preferred_term': YELLOW_HEADER,
    'value_list':                  YELLOW_HEADER,
    'codelist_submission_value':   YELLOW_HEADER,
    'subset_codelist':             YELLOW_HEADER,
    'vlm_target':                  YELLOW_HEADER,
    # Reference-side enrichment from SDTM_Domain_Metadata — green
    'observation_class':           GREEN_HEADER,
}

V_WIDTHS = {
    'ds_id':                       18,
    'variable_name':               16,
    'bc_id':                       12,
    'domain':                      8,
    'observation_class':           16,
    'role':                        12,
    'data_type':                   12,
    'mandatory_variable':          10,
    'mandatory_value':             10,
    'origin_type':                 12,
    'origin_source':               14,
    'assigned_term_value':         28,
    'assigned_term_concept_id':    14,
    'assigned_term_preferred_term':35,
    'value_list':                  50,
    'codelist_concept_id':         14,
    'codelist_submission_value':   18,
    'subset_codelist':             14,
    'vlm_target':                  10,
}

write_sheet(ws_v, variables_out, V_FILLS, V_WIDTHS)
print(f'Variables: {len(variables_out):,} rows x {len(variables_out.columns)} cols')

In [ ]:
wb.save(OUTPUT_FILE)
print(f'\nWritten: {OUTPUT_FILE}')
print(f'File size: {OUTPUT_FILE.stat().st_size / 1024:.0f} KB')

## 5. Summary

In [ ]:
print('=== DSS_Variables_View summary ===')
print(f'Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')
print()
print(f'Variables: {len(variables_out):>6,} rows x {len(variables_out.columns):>2} cols')
print()
with_cl = variables_out['codelist_concept_id'] != ''
no_cl   = ~with_cl
pinned_mask = variables_out['assigned_term_value'] != ''
vl_mask     = variables_out['value_list'] != ''

print('With codelist binding (binding-mode partition):')
print(f'  pinned:                {(with_cl &  pinned_mask).sum():>6,}')
print(f'  value_list-restricted: {(with_cl & ~pinned_mask &  vl_mask).sum():>6,}')
print(f'  bare codelist:         {(with_cl & ~pinned_mask & ~vl_mask).sum():>6,}')
print(f'  subtotal:              {with_cl.sum():>6,}')
print()
print('Without codelist binding:')
print(f'  pinned (no codelist):  {(no_cl &  pinned_mask).sum():>6,}')
print(f'  value_list only:       {(no_cl & ~pinned_mask &  vl_mask).sum():>6,}')
print(f'  no constraint at all:  {(no_cl & ~pinned_mask & ~vl_mask).sum():>6,}')
print(f'  subtotal:              {no_cl.sum():>6,}')
print()
print(f'Total:                   {len(variables_out):>6,}')
